In [7]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
import bs4
import requests
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv, dotenv_values

load_dotenv(override=True)
env_dict = dotenv_values(".env")
print("--- Contents of .env file ---")
for key, value in env_dict.items():
    # Masking the value so you don't accidentally print your full API key to your screen!
    masked_value = f"{value[:5]}...{value[-4:]}" if len(value) > 10 else value
    print(f"{key}: {masked_value}")

--- Contents of .env file ---
OPENAI_API_KEY: sk-pr...-PAA
GOOGLE_API_KEY: AQ.Ab...dTQA
TAVILY_API_KEY: tvly-...keLk
USER_AGENT: rag-f.../1.0
LANGSMITH_TRACING: true
LANGSMITH_ENDPOINT: https....com
LANGSMITH_API_KEY: lsv2_...2276
LANGSMITH_PROJECT: langc...demy
HF_TOKEN: hf_vG...sddc
LANGCHAIN_TRACING_V2: true
LANGCHAIN_API_KEY: lsv2_...13cd
LANGCHAIN_PROJECT: langc...demo


In [8]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
    )
),
)
blog_docs = loader.load()

#split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000, chunk_overlap=50
)

#make splits
splits = text_splitter.split_documents(blog_docs)

#index
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma 

# vectorstore = Chroma.from_documents(
#     documents=splits,
#     embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))

hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# 3. Create your Chroma vector store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=hf_embeddings
)
retriever = vectorstore.as_retriever()



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

# 2. Build the chain (Swapped ChatOpenAI for ChatGoogleGenerativeAI)

local_llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model", # LM Studio automatically uses whatever model is loaded in the server
    temperature=0
)

generate_queries = (
    prompt_perspectives 
    | local_llm 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

# 4. Run it!
questions = generate_queries.invoke({"question": "What is an LLM agent?"})
print(questions)

['Explain the concept of an LLM agent in detail.', 'How do Large Language Model agents function or operate?', 'What are some real-world use cases or applications for implementing LLM agents?', 'Describe the technical architecture and core components (such as planning, memory, or tool usage) that define an LLM agent.', 'What capabilities allow an AI system to autonomously complete complex, multi-step tasks?']


In [9]:
from langchain_core.load import dumps, loads

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

#retrieve 
question = "what is task decomposition for llm agents"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})
len(docs)

/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/997358273.py:10: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]
/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/997358273.py:10: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


8

In [11]:
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model", # LM Studio automatically uses whatever model is loaded in the server
    temperature=0
)

final_rag_chain = (
    {"context": retrieval_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})


/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/997358273.py:10: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


''

In [12]:
from langchain_core.prompts import ChatPromptTemplate

# RAG-Fusion: Related
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_queries = (
    prompt_rag_fusion 
    | ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model", # LM Studio automatically uses whatever model is loaded in the server
    temperature=0)
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

NameError: name 'prompt_rag_fusion' is not defined

In [17]:
from langchain_core.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/2443664460.py:26: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


11

In [5]:
from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

NameError: name 'retrieval_chain_rag_fusion' is not defined

In [ ]:
##Decomposition 

In [1]:
from langchain_core.prompts import ChatPromptTemplate

# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


llm = ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model") # LM Studio automatically uses whatever model is loaded in the server
    
generate_queries_decomposition = ( prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n")))

# Run
question = "What are the main components of an LLM-powered autonomous agent system?"
questions = generate_queries_decomposition.invoke({"question":question})

In [12]:
questions

['1. What is the role of a planning module in an LLM-powered autonomous agent system? (Focuses on decision-making and control flow)',
 '2. How do LLMs integrate memory components (e.g., vector databases, short-term/long-term memory) to maintain state within an autonomous agent? (Focuses on context management)',
 '3. What are the technical requirements for enabling tool use and external API interactions within a large language model agent framework? (Focuses on action capability and system integration)']

In [13]:
# Prompt
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [14]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

def format_qa_pair(question, answer):
    """ Format Q and A pair """
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

llm = ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model") 

q_a_pairs = ""
for q in questions:
    rag_chain = (
    {"context": itemgetter("question") | retriever, 
     "question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs")} 
    | decomposition_prompt
    | llm
    | StrOutputParser())

    answer = rag_chain.invoke({"question":q,"q_a_pairs":q_a_pairs})
    q_a_pair = format_qa_pair(q,answer)
    q_a_pairs = q_a_pairs + "\n---\n"+  q_a_pair

In [15]:
answer

'The ability for an LLM agent to use external tools and APIs transforms it from a mere text predictor into an autonomous, action-capable system. The technical requirements span three main layers: **The Prompt Interface (Input)**, **The Runtime Backend (Execution)**, and **The Control Flow Logic (Orchestration)**.\n\nHere is a detailed breakdown of the technical requirements for enabling robust tool use and external API interactions.\n\n---\n\n### 🛠️ I. The Architectural Requirement: Implementing the ReAct Loop\n\nThe foundational requirement is establishing a rigid, structured operational cycle that manages state through actions and observations. This pattern is known as **ReAct (Reasoning + Action)**.\n\n**Technical Mechanism:**\n1.  **Thought ($\\text{LLM}$):** The LLM generates an internal reasoning step: *What* needs to be done and *Why*.\n2.  **Action ($\\text{LLM}$ $\\rightarrow$ System):** Based on the thought, the LLM must generate a structured output specifying which function/

In [24]:
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Initialize the LangSmith client
client = Client()

# Pull the prompt directly via LangSmith
prompt_rag = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

def retrieve_and_rag(question, prompt_rag, sub_question_generator_chain):
    """ RAG on each sub question """
     
    # Use our decomposition / 
    sub_questions = sub_question_generator_chain.invoke({"question":question})
    
    # Initialize a list to hold RAG chain results
    rag_results = []
    
    for sub_question in sub_questions:
        
        # Retrieve documents for each sub-question
        retrieved_docs = retriever.invoke(sub_question)
        
        # Use retrieved documents and sub-question in RAG chain
        answer = (prompt_rag | llm | StrOutputParser()).invoke({"context": retrieved_docs, 
                                                                "question": sub_question})
        rag_results.append(answer)
    
    return rag_results,sub_questions

# Wrap the retrieval and RAG process in a RunnableLambda for integration into a chain
answers, questions = retrieve_and_rag(question, prompt_rag, generate_queries_decomposition)



In [25]:
def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

context = format_qa_pairs(questions, answers)

# Prompt
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":context,"question":question})

'Based on the provided information, an LLM-powered autonomous agent system is a complex architecture that moves beyond simply using a large language model as a standalone chatbot. Instead, it integrates multiple specialized external components and sophisticated control flow mechanisms to grant the model capabilities of memory, action, and self-improvement.\n\nThe main components can be categorized into three functional pillars: **Memory**, **Action**, and **Intelligence/Control Flow**.\n\n### 🧠 1. Intelligence & Control Loop (Planning and Reasoning)\nThis is the "brain" that dictates *how* the agent thinks, plans, and corrects itself. The goal of this component is to achieve self-correction and complex multi-step reasoning:\n\n*   **ReAct (Reasoning + Action):** This foundational loop structures the agent\'s output into explicit, visible steps: `Thought` (internal reasoning), followed by an `Action` (what it should do), and concluding with an `Observation` (the result of that action).\

In [30]:
import pprint

pprint.pprint(final_rag_chain.invoke({"context":context,"question":question}))

('The architecture of a modern LLM-powered autonomous agent is highly modular, '
 'moving far beyond simple prompt-response generation. It requires several '
 'interconnected external components that enable it to maintain state, take '
 'action in the real world, and self-correct its strategy.\n'
 '\n'
 'Based on these specialized functions, there are three main structural '
 'pillars: **Memory Systems**, **Action Systems (Tools)**, and the central '
 '**Control/Reasoning Loop**.\n'
 '\n'
 '---\n'
 '\n'
 '### 🧠 1. Memory System (Context & Knowledge Retention)\n'
 '\n'
 'This component addresses the fundamental limitation of LLMs—their finite '
 'context window—by providing external, long-term memory capabilities.\n'
 '\n'
 '*   **Function:** To retain knowledge, conversation history, and proprietary '
 'data over extended periods, simulating "infinite" memory retention.\n'
 "*   **Mechanism:** Agents do not rely on the model's internal parameters; "
 'instead, they utilize **external v

In [32]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)


In [33]:

llm = ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model") 

generate_queries_step_back = prompt | llm | StrOutputParser()

question = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": question})

'What are common strategies for handling complex problems?'

In [36]:
# Response prompt 
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        # Retrieve context using the normal question
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve context using the step-back question
        "step_back_context": generate_queries_step_back | retriever,
        # Pass on the question
        "question": lambda x: x["question"],
    }
    | response_prompt
    | ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", temperature=0)
    | StrOutputParser()
)

chain.invoke({"question": question})

'Task decomposition is a fundamental capability in an LLM-powered autonomous agent system, serving as the mechanism by which'

In [37]:
pprint.pprint(chain.invoke({"question": question}))

''


In [40]:
###HyDe 

from langchain_core.prompts import ChatPromptTemplate
#hyde document generation
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_docs_for_retrieval = (
    prompt_hyde | ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", temperature=0) | StrOutputParser()
)

#run
question = "What is task decomposition for LLM Agents"
generate_docs_for_retrieval.invoke({"question":question})

'## Task Decomposition for Large Language Model Agents: A Framework for Complex Goal Achievement\n\n**Abstract:** As Large Language Model (LLM) agents are deployed to handle increasingly complex, multi-faceted real-world tasks, the inherent limitation of single-pass, monolithic reasoning becomes a critical bottleneck. This passage defines and elaborates on task decomposition—a crucial architectural pattern that enables LLM agents to manage high levels of complexity by systematically breaking down overarching goals into manageable, sequential sub-tasks.\n\n***\n\n### 1. Introduction and Definition\n\nIn the context of autonomous AI systems, an **LLM Agent** is defined as a system comprising a core generative model (the LLM) coupled with external tools, memory modules, and a planning mechanism that allows it to interact with its environment to achieve specified objectives. When faced with a complex prompt—for example, "Analyze the market impact of quantum computing on semiconductor manuf

In [41]:
retrieval_chain = generate_docs_for_retrieval | retriever 
retrieved_docs = retrieval_chain.invoke({"question":question})
retrieved_docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refi

In [43]:
# rag
template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

answer = final_rag_chain.invoke({"context":retrieved_docs,"question":question})

pprint.pprint(answer)

('Task decomposition for LLM Agents refers to the process by which an agent '
 'breaks down a large, complex task into smaller, more manageable subgoals. '
 'This capability is essential for enabling the agent to handle intricate '
 'problems efficiently.\n'
 '\n'
 'The provided context details several ways that agents can perform task '
 'decomposition:\n'
 '\n'
 '### 1. Core Concepts\n'
 '*   **Goal:** To break down complicated tasks into steps, allowing the agent '
 'to manage complexity and improve its chances of success.\n'
 '*   **Self-Improvement:** Decomposition is linked to self-reflection and '
 'refinement, where the agent learns from mistakes made in smaller steps.\n'
 '\n'
 '### 2. Specific Decomposition Techniques\n'
 '\n'
 '**A. Prompting/Reasoning Techniques:**\n'
 '\n'
 '*   **Chain of Thought (CoT):** This standard technique prompts the model to '
 '"think step by step." It utilizes test-time computation to decompose hard '
 "tasks into simpler, manageable steps and r

Failed to refresh cache entry rlm/rag-prompt: Connection error caused failure to GET /commits/rlm/rag-prompt/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/rlm/rag-prompt/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 8] nodename nor servname provided, or not known)"))'))
Content-Length: None
API Key: lsv2_********************************************76
Failed to refresh cache entry rlm/rag-prompt: Connection error caused failure to GET /commits/rlm/rag-prompt/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/rlm/rag-prompt/latest (Caused by NameResolutionError("HTTPSConnection(ho